# 14 — Train all-class cached RGB + MediaPipe pose fusion

Notebook này dùng hai nhánh: **MediaPipe Graph → Spatial Transformer → Temporal Transformer** và **cached frozen VideoMAE V2 tokens → RGB Temporal Transformer**. Hai embedding được ghép bằng một gated late-fusion head. VideoMAE không được tải hoặc chạy lại trong lúc train.

Số lớp được đọc động từ manifest `all_eligible`. Test chỉ được chuẩn bị sau khi validation đã chọn checkpoint tốt nhất. Kết quả này không được so trực tiếp với baseline pose top‑70 vì data contract và số lớp khác nhau.


In [ ]:
#@title Configuration
PROJECT_GIT_REF = 'feat/kaggle-vsl-mediapipe-graph'  #@param {type:'string'}
DATA_CONTRACT_NAME = 'all_eligible_latest_min2_val0p2_seed42'  #@param {type:'string'}
RUN_NAME = 'videomaev2_pose_fusion_all_eligible_v1'  #@param {type:'string'}
MAX_EPOCHS = 100  #@param {type:'integer'}
BATCH_SIZE = 16  #@param {type:'integer'}
LEARNING_RATE = 0.0003  #@param {type:'number'}
WEIGHT_DECAY = 0.05  #@param {type:'number'}
LABEL_SMOOTHING = 0.10  #@param {type:'number'}
CLASS_BALANCE_POWER = 0.50  #@param {type:'number'}
WARMUP_EPOCHS = 5  #@param {type:'integer'}
EARLY_STOPPING_PATIENCE = 10  #@param {type:'integer'}
EARLY_STOPPING_MIN_DELTA = 0.002  #@param {type:'number'}
GRAPH_DIM = 96  #@param {type:'integer'}
TEMPORAL_DIM = 192  #@param {type:'integer'}
FUSION_DIM = 128  #@param {type:'integer'}
DROPOUT = 0.30  #@param {type:'number'}
MODALITY_DROPOUT = 0.15  #@param {type:'number'}
LOG_EVERY_BATCHES = 25  #@param {type:'integer'}
COPY_PACKS_TO_LOCAL = True  #@param {type:'boolean'}
RESUME = True  #@param {type:'boolean'}
RUN_TEST_AFTER_TRAINING = True  #@param {type:'boolean'}
SEED = 42  #@param {type:'integer'}


In [ ]:
#@title Mount Drive and define artifacts
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/silent-signal-results/vsl_kaggle_rgb_pose')
SUBSET_ROOT = DRIVE_ROOT / 'subsets' / DATA_CONTRACT_NAME
MANIFEST = SUBSET_ROOT / 'manifest.csv'
LABELS = SUBSET_ROOT / 'labels.json'
POSE_PACK_DRIVE = SUBSET_ROOT / 'pose/mediapipe76_front.npz'
POSE_REPORT = SUBSET_ROOT / 'pose/mediapipe76_front.report.json'
RGB_PACK_DRIVE = SUBSET_ROOT / 'rgb/videomaev2_base_tokens_fp16.npz'
RGB_REPORT = SUBSET_ROOT / 'rgb/videomaev2_base_tokens_fp16.report.json'
RUN_ROOT = DRIVE_ROOT / 'runs' / RUN_NAME
FIGURES_ROOT = RUN_ROOT / 'figures'
LOCAL_REPO = Path('/content/silent-signal')
LOCAL_POSE_PACK = Path(f'/content/mediapipe76_front_{DATA_CONTRACT_NAME}.npz')
LOCAL_RGB_PACK = Path(f'/content/videomaev2_base_{DATA_CONTRACT_NAME}_fp16.npz')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
#@title Checkout code, install package and verify GPU
import subprocess, sys
if not (LOCAL_REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch',
                    'https://github.com/stillthethrone/silent-signal.git', str(LOCAL_REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'fetch', 'origin', PROJECT_GIT_REF], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'checkout', '-B', PROJECT_GIT_REF,
                    f'origin/{PROJECT_GIT_REF}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{LOCAL_REPO}[training]'], check=True)
commit = subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
import torch
print('Commit:', commit)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before training.')


In [ ]:
#@title Verify Drive artifacts and copy the two packs to local disk
import hashlib, json, shutil, time
from silent_signal.data.manifest import read_manifest
from silent_signal.data.keypoint_pack import read_packed_keypoints
from silent_signal.data.rgb_feature_pack import read_rgb_feature_pack
from silent_signal.pose.cache import sha256_file
required = (MANIFEST, LABELS, POSE_PACK_DRIVE, POSE_REPORT, RGB_PACK_DRIVE, RGB_REPORT)
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing notebook 13 artifacts: {missing}')
records = tuple(sorted(read_manifest(MANIFEST), key=lambda row: row.sample_id))
pose_report = json.loads(POSE_REPORT.read_text(encoding='utf-8'))
rgb_report = json.loads(RGB_REPORT.read_text(encoding='utf-8'))
manifest_sha = sha256_file(MANIFEST)
pose_pack_sha = sha256_file(POSE_PACK_DRIVE)
rgb_pack_sha = sha256_file(RGB_PACK_DRIVE)
if pose_report.get('status') != 'complete' or rgb_report.get('status') != 'complete':
    raise RuntimeError('Notebook 13 did not finish both packed artifacts.')
if pose_report.get('manifest_sha256') != manifest_sha or rgb_report.get('manifest_sha256') != manifest_sha:
    raise RuntimeError('Manifest and feature packs do not share the same data identity; rerun notebook 13.')
if pose_report.get('output_sha256') != pose_pack_sha:
    raise RuntimeError('Pose pack failed SHA-256 verification.')
if rgb_report.get('output_sha256') != rgb_pack_sha:
    raise RuntimeError('RGB feature pack failed SHA-256 verification.')
if COPY_PACKS_TO_LOCAL:
    def copy_verified(source, destination, expected):
        if destination.is_file() and sha256_file(destination) == expected:
            print('Reuse local:', destination)
            return destination
        partial = destination.with_suffix(destination.suffix + '.partial')
        partial.unlink(missing_ok=True)
        shutil.copyfile(source, partial)
        if sha256_file(partial) != expected:
            partial.unlink(missing_ok=True)
            raise RuntimeError(f'Copy failed SHA-256 verification: {source}')
        partial.replace(destination)
        print('Copied:', destination)
        return destination
    POSE_PACK = copy_verified(POSE_PACK_DRIVE, LOCAL_POSE_PACK, pose_pack_sha)
    RGB_PACK = copy_verified(RGB_PACK_DRIVE, LOCAL_RGB_PACK, rgb_pack_sha)
else:
    POSE_PACK, RGB_PACK = POSE_PACK_DRIVE, RGB_PACK_DRIVE
pose = read_packed_keypoints(POSE_PACK)
rgb = read_rgb_feature_pack(RGB_PACK)
expected_ids = tuple(row.sample_id for row in records)
assert tuple(sorted(pose.sample_ids)) == expected_ids
assert rgb.sample_ids == expected_ids
assert rgb.metadata.get('manifest_sha256') == manifest_sha
CLASS_COUNT = len({row.class_index for row in records})
print(f'Data ready: {len(records):,} samples | {CLASS_COUNT} classes | RGB {rgb.features.shape}')
del pose, rgb
import gc; gc.collect()


In [ ]:
#@title Train gated RGB + pose fusion
if RESUME and (RUN_ROOT / 'last_checkpoint.pt').is_file():
    print('Resume is enabled. This only works when data, code commit and every hyperparameter are unchanged.')
elif RESUME:
    print('No checkpoint found; starting a new run in', RUN_ROOT)
command = [sys.executable, '-u', '-m', 'silent_signal.cli.train_rgb_pose_fusion',
           '--manifest', str(MANIFEST), '--keypoints', str(POSE_PACK),
           '--rgb-features', str(RGB_PACK), '--output-root', str(RUN_ROOT),
           '--epochs', str(MAX_EPOCHS), '--batch-size', str(BATCH_SIZE),
           '--learning-rate', str(LEARNING_RATE), '--weight-decay', str(WEIGHT_DECAY),
           '--label-smoothing', str(LABEL_SMOOTHING),
           '--class-balance-power', str(CLASS_BALANCE_POWER),
           '--warmup-epochs', str(WARMUP_EPOCHS), '--patience', str(EARLY_STOPPING_PATIENCE),
           '--min-delta', str(EARLY_STOPPING_MIN_DELTA), '--seed', str(SEED), '--workers', '2',
           '--progress-every-batches', str(LOG_EVERY_BATCHES), '--device', 'cuda',
           '--project-commit', commit, '--graph-dim', str(GRAPH_DIM), '--graph-blocks', '2',
           '--spatial-layers', '2', '--spatial-heads', '4',
           '--temporal-dim', str(TEMPORAL_DIM), '--temporal-layers', '2', '--temporal-heads', '4',
           '--pose-dropout', str(DROPOUT), '--fusion-dim', str(FUSION_DIM),
           '--rgb-layers', '1', '--rgb-heads', '4', '--fusion-dropout', str(DROPOUT),
           '--modality-dropout', str(MODALITY_DROPOUT)]
if not RESUME:
    command.append('--no-resume')
if RUN_TEST_AFTER_TRAINING:
    command.append('--run-test')
print('+', ' '.join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
#@title Training curves and final metrics
import matplotlib.pyplot as plt
import numpy as np
report = json.loads((RUN_ROOT / 'report.json').read_text(encoding='utf-8'))
history = json.loads((RUN_ROOT / 'history.json').read_text(encoding='utf-8'))
epochs = [row['epoch'] for row in history]
best = report['best_epoch']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0,0].plot(epochs, [r['train_loss'] for r in history], label='train')
axes[0,0].plot(epochs, [r['validation_loss'] for r in history], label='validation')
axes[0,0].set_title('Cross-entropy loss'); axes[0,0].legend()
axes[0,1].plot(epochs, [r['train_top1'] for r in history], label='train')
axes[0,1].plot(epochs, [r['validation_top1'] for r in history], label='validation')
axes[0,1].set_title('Top-1'); axes[0,1].legend()
axes[0,2].plot(epochs, [r['validation_top5'] for r in history])
axes[0,2].set_title('Validation Top-5')
axes[1,0].plot(epochs, [r['validation_macro_f1'] for r in history])
axes[1,0].set_title('Validation Macro-F1')
axes[1,1].plot(epochs, [r['train_top1']-r['validation_top1'] for r in history])
axes[1,1].axhline(0, color='black', lw=0.8); axes[1,1].set_title('Generalization gap')
axes[1,2].plot(epochs, [r['learning_rate'] for r in history], color='purple')
axes[1,2].set_title('Learning rate')
for axis in axes.flat:
    axis.axvline(best, color='gray', linestyle=':', label='best')
    axis.set_xlabel('epoch'); axis.grid(alpha=0.25)
fig.suptitle(f'Cached VideoMAE V2 + MediaPipe fusion — {CLASS_COUNT} classes — best epoch {best}')
fig.tight_layout(); fig.savefig(FIGURES_ROOT / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()
print(json.dumps({k: report[k] for k in ('parameters','epochs_completed','best_epoch','best_validation_loss','stopped_early','evaluation')}, ensure_ascii=False, indent=2))


In [ ]:
#@title Per-class F1, normalized confusion and top error directions
import csv
for split in ('validation', 'test'):
    per_class_path = RUN_ROOT / f'per_class_{split}.csv'
    confusion_path = RUN_ROOT / f'confusion_{split}.csv'
    if not per_class_path.is_file() or not confusion_path.is_file():
        continue
    with per_class_path.open(encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    rows.sort(key=lambda row: float(row['f1']))
    shown = rows[:30] + rows[-20:]
    colors = ['#d62728' if float(r['f1']) < 0.4 else '#ffb000' if float(r['f1']) < 0.8 else '#2ca02c' for r in shown]
    figure, axis = plt.subplots(figsize=(10, 14))
    axis.barh(range(len(shown)), [float(r['f1']) for r in shown], color=colors)
    axis.set_yticks(range(len(shown)), [f"{r['gloss']} (n={r['support']})" for r in shown], fontsize=7)
    axis.set_xlim(0,1); axis.set_xlabel('F1'); axis.set_title(f'30 lowest + 20 highest class F1 — {split}')
    figure.tight_layout(); figure.savefig(FIGURES_ROOT / f'per_class_f1_{split}.png', dpi=160)
    plt.show()

    matrix = np.loadtxt(confusion_path, delimiter=',', dtype=np.int64)
    normalized = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    figure, axis = plt.subplots(figsize=(12, 11))
    image = axis.imshow(normalized, cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    axis.set(title=f'Normalized confusion — {split} ({CLASS_COUNT} classes)', xlabel='Predicted class index', ylabel='True class index')
    figure.colorbar(image, ax=axis); figure.tight_layout()
    figure.savefig(FIGURES_ROOT / f'confusion_{split}.png', dpi=180); plt.show()

    names = [None] * CLASS_COUNT
    for row in rows:
        names[int(row['class_index'])] = row['gloss']
    errors = sorted(((int(matrix[i,j]), i, j) for i in range(CLASS_COUNT) for j in range(CLASS_COUNT) if i != j and matrix[i,j] > 0), reverse=True)[:20]
    print(f'\nTop confusion directions — {split}:')
    for count, true_index, pred_index in errors:
        print(f'{count:>3} | {names[true_index]} → {names[pred_index]}')


## Interpretation contract

- Checkpoint được chọn bằng validation loss; test không tham gia lựa chọn.
- Class-balanced loss giảm ảnh hưởng của chênh lệch số mẫu giữa toàn bộ từ.
- Modality dropout buộc mô hình không phụ thuộc tuyệt đối vào RGB hoặc pose.
- Muốn chứng minh fusion tốt hơn pose-only, cần chạy thêm pose-only trên **chính manifest all_eligible này**; không dùng kết quả top‑70 làm đối chứng trực tiếp.
- Giữ nguyên `RUN_NAME` chỉ khi resume đúng cùng dữ liệu, commit và tham số. Khi đổi cấu hình, hãy đặt `RUN_NAME` mới.
